In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean visualization style
sns.set_theme(style="whitegrid")

# 1. Load the dataset
df = pd.read_csv('../dataset/fake_job/fake_job_postings.csv')

# 2. Basic dataset inspection
print("=" * 40)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("=" * 40)

# Display first few rows
df.head(3)

Dataset Shape: 17880 rows, 18 columns


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0


In [3]:
# 1. Check target class distribution (0 = Real, 1 = Fraudulent)
target_col = 'fraudulent'
counts = df[target_col].value_counts()
pcts = df[target_col].value_counts(normalize=True) * 100

print("\n--- Target Distribution ---")
print(f"Real Job Postings (0) : {counts[0]} ({pcts[0]:.2f}%)")
print(f"Fake Job Postings (1) : {counts[1]} ({pcts[1]:.2f}%)")

# 2. Fill missing text fields with blank spaces
text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
for col in text_cols:
    df[col] = df[col].fillna('')

# 3. Create unified 'full_text' feature combining all document sections
df['full_text'] = (
    df['title'] + " " + 
    df['company_profile'] + " " + 
    df['description'] + " " + 
    df['requirements'] + " " + 
    df['benefits']
)

# 4. Compute basic text stats
df['word_count'] = df['full_text'].apply(lambda x: len(x.split()))

print("\n--- Word Count Summary by Class ---")
print(df.groupby('fraudulent')['word_count'].describe())

# 5. Save the processed CSV ready for TF-IDF / Baseline Model Training
df.to_csv('../dataset/fake_job/processed_fake_job_postings.csv', index=False)
print("\n✅ Saved processed dataset to 'dataset/fake_job/processed_fake_job_postings.csv'")


--- Target Distribution ---
Real Job Postings (0) : 17014 (95.16%)
Fake Job Postings (1) : 866 (4.84%)

--- Word Count Summary by Class ---
              count        mean         std  min    25%    50%     75%     max
fraudulent                                                                    
0           17014.0  379.477313  206.713703  7.0  235.0  355.0  496.00  2118.0
1             866.0  282.332564  203.864781  2.0  133.0  224.5  384.25  1338.0

✅ Saved processed dataset to 'dataset/fake_job/processed_fake_job_postings.csv'


In [4]:
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# 0. Automatically create output directory
os.makedirs('../models/fake_job', exist_ok=True)

# 1. Feature and Target Definition
X = df['full_text']
y = df['fraudulent']

# 2. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Samples: {len(X_train)} | Testing Samples: {len(X_test)}")

# 3. Memory-Efficient TF-IDF Vectorization (Unigrams only, max 3000 features)
print("\nExtracting features using TF-IDF...")
tfidf = TfidfVectorizer(
    max_features=3000, 
    stop_words='english', 
    ngram_range=(1, 1), 
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Save Vectorizer
joblib.dump(tfidf, '../models/fake_job/tfidf_vectorizer.pkl')
print("✅ Saved TF-IDF Vectorizer to 'models/fake_job/tfidf_vectorizer.pkl'")

# 4. Fast & Lightweight Classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=500, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1)
}

results = []

for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train_tfidf, y_train)
    
    # Predictions
    y_pred = clf.predict(X_test_tfidf)
    y_proba = clf.predict_proba(X_test_tfidf)[:, 1] if hasattr(clf, "predict_proba") else y_pred
    
    # Calculate performance metrics
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    
    results.append({
        "Model": name,
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc, 4)
    })
    
    # Save Model
    filename = f"../models/fake_job/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(clf, filename)

# Display Comparative Results
results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("     MODULE 1: MODEL COMPARISON REPORT")
print("="*50)
print(results_df.to_string(index=False))

Training Samples: 14304 | Testing Samples: 3576

Extracting features using TF-IDF...
✅ Saved TF-IDF Vectorizer to 'models/fake_job/tfidf_vectorizer.pkl'

Training Logistic Regression...

Training Multinomial Naive Bayes...

Training Random Forest...

     MODULE 1: MODEL COMPARISON REPORT
                  Model  Precision  Recall  F1-Score  ROC-AUC
    Logistic Regression     0.6597  0.9075    0.7640   0.9875
Multinomial Naive Bayes     0.9839  0.3526    0.5191   0.9479
          Random Forest     0.5971  0.7110    0.6491   0.9714


In [6]:
import numpy as np

# Load trained Logistic Regression model and vectorizer
lr_model = joblib.load('../models/fake_job/logistic_regression.pkl')
vectorizer = joblib.load('../models/fake_job/tfidf_vectorizer.pkl')

feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = lr_model.coef_[0]

def predict_and_explain_job(text):
    # Vectorize input text
    text_tfidf = vectorizer.transform([text])
    
    # Prediction and Probability
    pred = lr_model.predict(text_tfidf)[0]
    prob = lr_model.predict_proba(text_tfidf)[0][1] # Probability of being Fake (1)
    
    # Feature Importance (Get terms in input text that contributed most positively to 'Fake')
    non_zero_indices = text_tfidf.nonzero()[1]
    word_weights = []
    
    for idx in non_zero_indices:
        word = feature_names[idx]
        weight = coefficients[idx]
        word_weights.append((word, weight))
    
    # Sort words by highest positive weight towards fraud classification
    suspicious_indicators = sorted([item for item in word_weights if item[1] > 0], key=lambda x: x[1], reverse=True)[:5]
    
    print("\n" + "="*50)
    print("      EXPLAINABLE AI FRAUD DETECTION RESULT")
    print("="*50)
    print(f"Status           : {'🚨 FRAUDULENT / FAKE JOB' if pred == 1 else '✅ LEGITIMATE JOB'}")
    print(f"Fraud Risk Score : {prob * 100:.2f}%")
    
    if suspicious_indicators:
        print("\nTop Suspicious Indicators Found:")
        for word, score in suspicious_indicators:
            print(f" - '{word}' (Risk Weight: +{score:.3f})")
    else:
        print("\nNo high-risk fraud triggers detected.")
    print("="*50)

# --- TEST WITH SAMPLE SKETCHY TEXT ---
sample_fake_job = """
Earn $5000 a week working from home! No experience required. Urgent hiring for entry level position.
We require a wire transfer or processing fee for equipment startup. Contact via personal Telegram account immediately.
"""

predict_and_explain_job(sample_fake_job)


      EXPLAINABLE AI FRAUD DETECTION RESULT
Status           : 🚨 FRAUDULENT / FAKE JOB
Fraud Risk Score : 96.05%

Top Suspicious Indicators Found:
 - 'earn' (Risk Weight: +4.211)
 - 'entry' (Risk Weight: +3.229)
 - 'urgent' (Risk Weight: +1.738)
 - 'hiring' (Risk Weight: +1.705)
 - 'home' (Risk Weight: +1.391)
